# RS-VLM — Phase 1: MAE Pretraining

Trains the **CNN-ViT hybrid encoder** on EuroSAT using Masked Autoencoding.

- **What trains:** CNNStem + ViTBody + GSDAdapter + MAEDecoder (decoder discarded after)
- **Dataset:** EuroSAT RGB (27k images) — already on Drive at `rs_vlm_data/EuroSAT data/`
- **Objective:** reconstruct 75% masked CNN feature map patches
- **Output:** `encoder_final.pt` → saved to Drive

## 0. Check GPU

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1. Mount Drive & verify dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

EUROSAT_DIR = '/content/drive/MyDrive/rs_vlm_data/EuroSAT'
CKPT_DIR    = '/content/drive/MyDrive/rs_vlm/checkpoints/phase1'

# verify the dataset folder is actually there
assert os.path.exists(EUROSAT_DIR), f"EuroSAT not found at: {EUROSAT_DIR}"
classes = os.listdir(EUROSAT_DIR)
print(f"EuroSAT found — {len(classes)} classes: {classes}")

# make sure checkpoint dir exists
os.makedirs(CKPT_DIR, exist_ok=True)
print(f"Checkpoint dir ready: {CKPT_DIR}")

## 2. Clone repo & install dependencies

In [ ]:
!git clone https://github.com/sharksurfauto-byte/rs_vlm.git /content/rs_vlm
%cd /content/rs_vlm

In [ ]:
!pip install -q -r requirements.txt
print('Dependencies installed.')

## 3. Smoke test — one forward pass before committing to 30 epochs

In [ ]:
import sys
sys.path.insert(0, '/content/rs_vlm')

import torch
from encoder.hybrid_encoder import HybridEncoder
from training.phase1_pretrain import MAEDecoder, mae_loss
from data.eurosat import mae_mask_patches

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

encoder = HybridEncoder(cnn_pretrained=False).to(device)
decoder = MAEDecoder().to(device)

dummy_imgs = torch.randn(2, 3, 224, 224).to(device)
dummy_gsd  = torch.tensor([10.0, 0.3]).to(device)

feat_map         = encoder.cnn_stem(dummy_imgs)
masked_map, mask = mae_mask_patches(feat_map, mask_ratio=0.75)
gsd_bias         = encoder.gsd_adapter(dummy_gsd)
tokens           = encoder.vit_body(masked_map, gsd_bias=gsd_bias)
reconstructed    = decoder(tokens)
loss             = mae_loss(feat_map.detach(), reconstructed, mask)

print(f'feat_map:      {feat_map.shape}')
print(f'tokens:        {tokens.shape}')
print(f'reconstructed: {reconstructed.shape}')
print(f'MAE loss:      {loss.item():.4f}')
assert reconstructed.shape == feat_map.shape
print('\nSmoke test passed!')

## 4. Also smoke test the real dataloader (reads one batch from Drive)

In [ ]:
from data.eurosat import get_eurosat_dataloader

loader = get_eurosat_dataloader(
    root=EUROSAT_DIR,
    train=True,
    batch_size=4,
    num_workers=0   # 0 workers for the quick test, avoids Colab multiproc issues
)

batch = next(iter(loader))
print(f"image shape:  {batch['image'].shape}")
print(f"gsd:          {batch['gsd']}")
print(f"labels:       {batch['label']}")
print(f"class names:  {batch['class_name']}")
print(f"Total batches: {len(loader)}  (batch_size=4)")
print('Dataloader smoke test passed!')

## 5. Run Phase 1 training

To **resume** from a checkpoint, set `resume_from` to a `.pt` path on Drive.

In [ ]:
from training.phase1_pretrain import train_phase1

encoder = train_phase1(
    config_path='configs/colab_config.yaml',
    resume_from=None,  # e.g. '/content/drive/MyDrive/rs_vlm/checkpoints/phase1/encoder_epoch010.pt'
)

## 6. Verify saved checkpoint

In [ ]:
import torch

ckpt_path = f'{CKPT_DIR}/encoder_final.pt'
ckpt = torch.load(ckpt_path, map_location='cpu')

print(f"Saved at epoch : {ckpt['epoch']}")
print(f"Final MAE loss : {ckpt['loss']:.4f}")
print(f"Keys           : {list(ckpt.keys())}")
print(f"\nPath for Phase 2 → phase1_checkpoint='{ckpt_path}'")